In [ ]:
#Luis Jose Gonzalez Montilla

In [ ]:
!pip install -q transformers datasets evaluate sentencepiece accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import pipeline
import evaluate
import time
import pandas as pd
import torch

device = 0 if torch.cuda.is_available() else -1
device


-1

In [ ]:
dataset = load_dataset("ccdv/arxiv-summarization")
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})

In [ ]:
sample = dataset["validation"][0]
for k, v in sample.items():
    print(f"=== {k.upper()} ===")
    print(v[:700], "...\n")


=== ARTICLE ===
the interest in anchoring phenomena and phenomena in confined nematic liquid crystals has largely been driven by their potential use in liquid crystal display devices . 
 the twisted nematic liquid crystal cell serves as an example . 
 it consists of a nematic liquid crystal confined between two parallel walls , both providing homogeneous planar anchoring but with mutually perpendicular easy directions . in this case 
 the orientation of the nematic director is tuned by the application of an external electric or magnetic field . 
 a precise control of the surface alignment extending over large areas is decisive for the functioning of such devices . 
 most studies have focused on nematic liqu ...

=== ABSTRACT ===
we study the phase behavior of a nematic liquid crystal confined between a flat substrate with strong anchoring and a patterned substrate whose structure and local anchoring strength we vary . by first evaluating an effective surface free energy function charac

In [ ]:
MAX_SAMPLES = 10
eval_dataset = dataset["validation"].select(range(MAX_SAMPLES))
len(eval_dataset)


10

In [ ]:
!pip install rouge_score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a7a75e7a5db50081a8bdfcdbc85125e33fc13d595d37e98486bb6c847ffaa460
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
rouge = evaluate.load("rouge")
rouge


EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [ ]:
def run_summarization_experiment(
    model_name,
    dataset,
    text_column="article",
    summary_column="abstract",
    max_samples=10,
    batch_size=2,
    max_summary_length=128,
    min_summary_length=20,
    max_input_chars=2000,
):
    """
    Corre un experimento de summarization usando un pipeline de Hugging Face.
    Devuelve un diccionario con métricas ROUGE y tiempos.
    """
    print(f"\n========== MODELO: {model_name} ==========")
    print(f"Dispositivo: {'GPU' if device == 0 else 'CPU'}")

    # Crear pipeline de summarization
    summarizer = pipeline(
        "summarization",
        model=model_name,
        tokenizer=model_name,
        device=device
    )

    # Limitar samples
    n = min(max_samples, len(dataset))

    #  Recortamos el texto de entrada para no explotar el modelo
    texts = [
        dataset[i][text_column][:max_input_chars]
        for i in range(n)
    ]
    refs  = [dataset[i][summary_column] for i in range(n)]

    print(f"Evaluando con {n} ejemplos...")

    preds = []
    start_time = time.time()

    # Inferencia en batches
    for i in range(0, n, batch_size):
        batch_texts = texts[i:i+batch_size]

        outputs = summarizer(
            batch_texts,
            max_length=max_summary_length,
            min_length=min_summary_length,
            truncation=True
        )

        batch_summaries = [o["summary_text"] for o in outputs]
        preds.extend(batch_summaries)

    elapsed = time.time() - start_time

    # Calcular ROUGE
    metrics = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    metrics = {k: v * 100 for k, v in metrics.items()}  # en %

    print("\nMÉTRICAS ROUGE:")
    for k, v in metrics.items():
        print(f"{k}: {v:.2f}")

    print(f"\nTiempo total: {elapsed:.2f} s")
    print(f"Tiempo promedio por ejemplo: {elapsed / n:.2f} s/ejemplo")

    result = {
        "task": "summarization",
        "dataset": "ccdv/arxiv-summarization",
        "model_name": model_name,
        "num_samples": n,
        "rouge1": metrics.get("rouge1", None),
        "rouge2": metrics.get("rouge2", None),
        "rougeL": metrics.get("rougeL", None),
        "rougeLsum": metrics.get("rougeLsum", None),
        "total_time_sec": elapsed,
        "time_per_sample_sec": elapsed / n,
    }

    return result


In [ ]:
models_to_test = [
    "facebook/bart-base",
    "facebook/bart-large-cnn",
    "t5-small",
    "t5-base",
]



In [ ]:
results = []

for model_name in models_to_test:
    res = run_summarization_experiment(
        model_name=model_name,
        dataset=eval_dataset,
        text_column="article",
        summary_column="abstract",
        max_samples=MAX_SAMPLES,
        batch_size=2,
        max_summary_length=128,
        min_summary_length=20,
        max_input_chars=2000,
    )
    results.append(res)

results




========== MODELO: facebook/bart-base ==========
Dispositivo: CPU


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Evaluando con 10 ejemplos...


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MÉTRICAS ROUGE:
rouge1: 19.63
rouge2: 4.08
rougeL: 12.15
rougeLsum: 15.98

Tiempo total: 180.56 s
Tiempo promedio por ejemplo: 18.06 s/ejemplo


[{'task': 'summarization',
  'dataset': 'ccdv/arxiv-summarization',
  'model_name': 'facebook/bart-base',
  'num_samples': 10,
  'rouge1': np.float64(35.505576332762836),
  'rouge2': np.float64(8.496536019022907),
  'rougeL': np.float64(17.404046128613317),
  'rougeLsum': np.float64(26.362014439313043),
  'total_time_sec': 343.55870938301086,
  'time_per_sample_sec': 34.35587093830109},
 {'task': 'summarization',
  'dataset': 'ccdv/arxiv-summarization',
  'model_name': 'facebook/bart-large-cnn',
  'num_samples': 10,
  'rouge1': np.float64(23.47935594790174),
  'rouge2': np.float64(6.607313604483947),
  'rougeL': np.float64(14.607922981842764),
  'rougeLsum': np.float64(18.812293170284335),
  'total_time_sec': 238.38284420967102,
  'time_per_sample_sec': 23.838284420967103},
 {'task': 'summarization',
  'dataset': 'ccdv/arxiv-summarization',
  'model_name': 't5-small',
  'num_samples': 10,
  'rouge1': np.float64(17.423037146136565),
  'rouge2': np.float64(3.9287505749665925),
  'rougeL'

In [ ]:
df_results = pd.DataFrame(results)
df_results


,task,dataset,model_name,num_samples,rouge1,rouge2,rougeL,rougeLsum,total_time_sec,time_per_sample_sec
0,summarization,ccdv/arxiv-summarization,facebook/bart-base,10,35.505576,8.496536,17.404046,26.362014,343.558709,34.355871
1,summarization,ccdv/arxiv-summarization,facebook/bart-large-cnn,10,23.479356,6.607314,14.607923,18.812293,238.382844,23.838284
2,summarization,ccdv/arxiv-summarization,t5-small,10,17.423037,3.928751,11.319194,13.851993,56.479329,5.647933
3,summarization,ccdv/arxiv-summarization,t5-base,10,19.627464,4.077625,12.150530,15.977147,180.556462,18.055646


In [ ]:
df_results.to_csv("benchmark_resultados_summarization.csv", index=False)
print("Archivo guardado: benchmark_resultados_summarization.csv")


Archivo guardado: benchmark_resultados_summarization.csv


In [ ]:
test_model = "facebook/bart-large-cnn"

summarizer = pipeline(
    "summarization",
    model=test_model,
    tokenizer=test_model,
    device=device
)

texto_usuario = """
The connection between Semantic Networks and Intelligent Systems with Visual Interfaces constitutes a
 fundamental pillar in the evolution of human-machine interaction.
 Semantic networks provide the deep, contextual, and reasonable knowledge structure that endows
 a system with significant intelligence. Visual interfaces, in turn, provide the high-bandwidth
 communication channel necessary to make this knowledge comprehensible, navigable, and manipulable
 for the human mind.

This symbiosis transcends the mere visualization of data.
It is about the visual externalization of a shared mental model.
It allows users not only to consume information but also to understand the underlying relationships,
 question the system's inferences, and actively collaborate in the construction and refinement of
 knowledge. As we advance toward more complex AI systems, the combination of a semantic "brain"
 with a visual "body" will become not only desirable but essential for building systems that are
 truly intelligent, transparent, and ultimately, worthy of our trust.

"""

res = summarizer(
    texto_usuario,
    max_new_tokens=80,
    min_new_tokens=20,
    truncation=True
)[0]["summary_text"]

print("=== TEXTO ORIGINAL ===")
print(texto_usuario)

print("\n=== RESUMEN GENERADO ===")
print(res)


Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


=== TEXTO ORIGINAL ===

The connection between Semantic Networks and Intelligent Systems with Visual Interfaces constitutes a
 fundamental pillar in the evolution of human-machine interaction.
 Semantic networks provide the deep, contextual, and reasonable knowledge structure that endows
 a system with significant intelligence. Visual interfaces, in turn, provide the high-bandwidth
 communication channel necessary to make this knowledge comprehensible, navigable, and manipulable
 for the human mind.

This symbiosis transcends the mere visualization of data.
It is about the visual externalization of a shared mental model.
It allows users not only to consume information but also to understand the underlying relationships,
 question the system's inferences, and actively collaborate in the construction and refinement of
 knowledge. As we advance toward more complex AI systems, the combination of a semantic "brain"
 with a visual "body" will become not only desirable but essential for build

In [ ]:

example = eval_dataset[0]
article = example["article"][:2000]
reference_summary = example["abstract"]

generated = summarizer(
    article,
    max_length=128,
    min_length=20,
    truncation=True
)[0]["summary_text"]

print("=== TEXTO ORIGINAL (recortado) ===")
print(article[:1000], "...\n")

print("=== RESUMEN REAL (ABSTRACT DEL DATASET) ===")
print(reference_summary, "\n")

print("=== RESUMEN GENERADO POR EL MODELO ===")
print(generated)


=== TEXTO ORIGINAL (recortado) ===
the interest in anchoring phenomena and phenomena in confined nematic liquid crystals has largely been driven by their potential use in liquid crystal display devices . 
 the twisted nematic liquid crystal cell serves as an example . 
 it consists of a nematic liquid crystal confined between two parallel walls , both providing homogeneous planar anchoring but with mutually perpendicular easy directions . in this case 
 the orientation of the nematic director is tuned by the application of an external electric or magnetic field . 
 a precise control of the surface alignment extending over large areas is decisive for the functioning of such devices . 
 most studies have focused on nematic liquid crystals in contact with laterally uniform substrates . on the other hand substrate inhomogeneities 
 arise rather naturally as a result of surface treatments such as rubbing . 
 thus the nematic texture near the surface is in fact non - uniform . 
 this non - u

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results


,task,dataset,model_name,num_samples,rouge1,rouge2,rougeL,rougeLsum,total_time_sec,time_per_sample_sec
0,summarization,ccdv/arxiv-summarization,facebook/bart-base,10,35.505576,8.496536,17.404046,26.362014,343.558709,34.355871
1,summarization,ccdv/arxiv-summarization,facebook/bart-large-cnn,10,23.479356,6.607314,14.607923,18.812293,238.382844,23.838284
2,summarization,ccdv/arxiv-summarization,t5-small,10,17.423037,3.928751,11.319194,13.851993,56.479329,5.647933
3,summarization,ccdv/arxiv-summarization,t5-base,10,19.627464,4.077625,12.150530,15.977147,180.556462,18.055646


In [ ]:
df_results[['model_name', 'num_samples', 'rouge1', 'rouge2', 'rougeL', 'time_per_sample_sec']]


,model_name,num_samples,rouge1,rouge2,rougeL,time_per_sample_sec
0,facebook/bart-base,10,35.505576,8.496536,17.404046,34.355871
1,facebook/bart-large-cnn,10,23.479356,6.607314,14.607923,23.838284
2,t5-small,10,17.423037,3.928751,11.319194,5.647933
3,t5-base,10,19.627464,4.077625,12.150530,18.055646


In [ ]:
df_results.to_csv("benchmark_resultados_summarization.csv", index=False)
print("Archivo guardado: benchmark_resultados_summarization.csv")


Archivo guardado: benchmark_resultados_summarization.csv
